# Histopathologic Cancer Detection by Image Classification

The dataset used in this project consists of 96 × 96 pixel RGB digital histopathology images of lymph node tissue for the detection of metastatic cancer. It was sourced from the publicly available Kaggle competition “Histopathologic Cancer Detection.” The dataset contains approximately 220,000 images and is a subset of the original PCam (PatchCamelyon) dataset, with the main difference being that duplicate images have been removed to ensure data quality and consistency.
These images are extracted from larger histopathology slides obtained from real clinical cases and are used to analyze the presence of cancer cells in lymph nodes. Each image represents a small tissue patch taken from a much larger whole-slide image. The objective is to determine whether metastatic cancer cells are present in each patch.
The task is formulated as a binary classification problem, where each image is labeled as either containing metastasis (1) or not (0).
Due to time limitations and the computational complexity of training deep neural networks on very large datasets, only a subset of the available data was used in this project. Although the full dataset contains approximately 220,000 images, all images were first downloaded, and then 5,000 samples were randomly selected to create a manageable working subset. A custom script was written to correctly match each selected image with its corresponding label, ensuring proper alignment between the input images and their ground-truth binary annotations.

# Dataset:

Original Dataset:

https://www.kaggle.com/competitions/histopathologic-cancer-detection


# Setup

In [ ]:
import os
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from tqdm.notebook import tqdm

# import data

In [ ]:
# Upload Kaggle API Credentials

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp -r "/content/drive/MyDrive/Histopathologic Cancer Detection" /content/

In [ ]:
data_dir = "/content/drive/MyDrive/Histopathologic Cancer Detection"
img_dir = os.path.join(data_dir, "Images")

In [ ]:
labels_dir = os.path.join(data_dir, "Labels" ,)
matched_labels = os.path.join(labels_dir, "matched_labels.csv")
matched_labels_df = pd.read_csv(matched_labels)

#print(matched_labels_df ["label"].value_counts())

# Path

In [ ]:
# for extracting the labels. rune just once
data_dir = "/content/drive/MyDrive/Histopathologic Cancer Detection"
img_dir = os.path.join(data_dir, "Images")
# 1. path
labels_dir = os.path.join(data_dir, "Labels")
images_labels_dir = os.path.join(labels_dir, "images_labels.csv")
OUTPUT_EXCEL = os.path.join(labels_dir, "matched_labels.csv")

# 2. Load labels CSV
labels_df = pd.read_csv(images_labels_dir)

# 3. Read image filenames
image_files = [
    f for f in os.listdir(img_dir)
    if f.lower().endswith(".tif")]
print("Images found in folder:", len(image_files))

# 4. Extract image IDs (remove .tif)
image_ids = set(f.replace(".tif", "") for f in image_files)

# 5. Filter labels to only existing images
matched_df = labels_df[labels_df["id"].isin(image_ids)].copy()
print("Matched image-label pairs:", len(matched_df))

# 6. Add full image path
matched_df["image_path"] = matched_df["id"].apply(
    lambda x: os.path.join(img_dir, x + ".tif"))

# 7. Save to CSV
matched_df.to_csv(OUTPUT_EXCEL, index=False)
print("✅ CSV file saved to:")
print(OUTPUT_EXCEL)

In [ ]:
print(matched_labels_df ["label"].value_counts())

# Dataset Splitting (Train/Val/Test Split)

In [ ]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
train, temp_df = train_test_split(
    matched_labels_df,
    test_size=0.3,
    stratify=matched_labels_df["label"],
    random_state=42
)

# 15% val, 15% test
val, test= train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

print(f'Train: {len(train)}, Val:{len(val)}, Test{len(test)}')

Train: 3500, Val:750, Test750


## Check Image Size and Channel Information

In [ ]:
image_sizes = set()
image_shapes = set()

image_files = [
    f for f in os.listdir(img_dir)
    if f.lower().endswith(".tif")
]

for img_name in tqdm(image_files, desc="Checking image sizes and channels"):
    img_path = os.path.join(img_dir, img_name)

    with Image.open(img_path) as img:
        img_arr = np.array(img)

        # (width, height)
        image_sizes.add(img.size)

        # (H, W) or (H, W, C)
        image_shapes.add(img_arr.shape)

print("✅ Unique image sizes:", image_sizes)
print("✅ Unique image array shapes:", image_shapes)

# Visualization

In [ ]:
sample_df = train.sample(10, random_state=42)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

label_map = {0: "No Cancer", 1: "Cancer"}

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    img = Image.open(row["image_path"])
    ax.imshow(img)

    ax.set_title(
        f"Label: {label_map[row['label']]}",
        fontsize=18,
        fontweight="bold"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

# Augmentation

These transformations are written to augment and preprocess images before feeding them into the neural network.
The strong_transforms pipeline applies random geometric and intensity changes to improve model robustness and prevent overfitting during training.
The transforms pipeline performs only resizing and normalization, ensuring consistent and stable inputs during validation and testing.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(180),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Defining Dataset

In [ ]:
class HistopathologicDataset(Dataset):
    """
    PyTorch Dataset for Histopathologic Cancer Detection (Classification).
    Loads image and label pairs and returns them as tensors.
    """

    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        img = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform is not None:
            img = self.transform(img)

        return img, label

# Dataset for Train / Val / Test

This code creates separate datasets for training, validation, and testing with appropriate preprocessing. Strong augmentations are applied only to training data, while validation and test data use consistent, non-augmented transforms for fair evaluation.

In [ ]:
train_dataset = HistopathologicDataset(train, transform=train_transform)
val_dataset = HistopathologicDataset(val, transform=val_test_transform)
test_dataset = HistopathologicDataset(test, transform=val_test_transform)

## Check

In [ ]:
img, label = train_dataset[0]

print(img.shape)   # torch.Size([3, 96, 96])
print(label)       # 0 or 1

# DataLoader

This code wraps each dataset in a DataLoader to load data in mini-batches during training and evaluation. Training data is shuffled to improve learning, while validation and test data are kept in a fixed order for consistent evaluation.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Define Model

In [ ]:
import torchvision.models as models
class HistopathologicClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.model = models.resnet18(pretrained=True)
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = HistopathologicClassifier(num_classes=2)
model = model.to(device)

Using device: cpu


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 65.5MB/s]


## Check

We wrote this code to verify that the dataset and DataLoader are correctly configured and that the images have the expected shapes before training the model.

In [ ]:
# Create DataLoader (small batch size only for verification)
data_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# Print dataset and DataLoader sizes
print("Number of training samples:", len(train_dataset))
print("Number of batches:", len(data_loader))

# Get one batch to verify shapes
images, labels = next(iter(data_loader))
images = images.to(device)
labels = labels.to(device)

print("Images batch shape:", images.shape)
print("Labels batch shape:", labels.shape)

Number of training samples: 3500
Number of batches: 875
Images batch shape: torch.Size([4, 3, 96, 96])
Labels batch shape: torch.Size([4])


'\nThis code is used to verify that the Dataset and DataLoader are correctly configured\nfor the histopathologic cancer classification task. It ensures that image batches\nhave the expected shape (batch_size, 3, H, W) and that each image is associated\nwith a single binary label before starting the training process.\n'

In [ ]:
Y_pred = model(images)
print(Y_pred.shape)

torch.Size([4, 2])


# Train the model

In [ ]:
batch_size = 16
epochs = 20
lr = 0.0001

In [ ]:
#GPU OR CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#Loss function and optimizer:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Train Model

In [ ]:
from tqdm import tqdm
import torch

best_val_loss = float("inf")
checkpoint_path = "best_model.pth"

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(epochs):

    # TRAIN
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Train"):
        images = images.to(device)      # (B, 3, H, W)
        labels = labels.to(device)      # (B,)

        optimizer.zero_grad()

        # Forward
        outputs = model(images)         # (B, 2)

        # Loss
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        # Accuracy
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    avg_train_loss = running_train_loss / len(train_loader)
    train_acc = correct_train / total_train

    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)

    #  VALIDATION
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - Val"):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    avg_val_loss = running_val_loss / len(val_loader)
    val_acc = correct_val / total_val

    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)

     # SAVE BEST MODEL
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print("✅ Best model saved")

    # LOG
    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )

Epoch 1/20 - Val: 100%|██████████| 47/47 [00:16<00:00,  2.85it/s]


✅ Best model saved
Epoch [1/20] | Train Loss: 0.4879, Train Acc: 0.7780 | Val Loss: 0.4269, Val Acc: 0.8053


Epoch 2/20 - Val: 100%|██████████| 47/47 [00:16<00:00,  2.86it/s]


Epoch [2/20] | Train Loss: 0.4320, Train Acc: 0.8149 | Val Loss: 0.4319, Val Acc: 0.8013


Epoch 3/20 - Val: 100%|██████████| 47/47 [00:16<00:00,  2.81it/s]


...

Epoch 12/20 - Val: 100%|██████████| 47/47 [00:16<00:00,  2.85it/s]


Epoch [12/20] | Train Loss: 0.2775, Train Acc: 0.8851 | Val Loss: 0.3709, Val Acc: 0.8587


Epoch 13/20 - Val: 100%|██████████| 47/47 [00:16<00:00,  2.84it/s]


✅ Best model saved
Epoch [13/20] | Train Loss: 0.2934, Train Acc: 0.8854 | Val Loss: 0.2486, Val Acc: 0.9080


Epoch 14/20 - Train:  96%|█████████▋| 211/219 [03:48<00:08,  1.12s/it]

# Visualizing Loss and Accuracy

In [ ]:
epochs = range(1, len(train_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ===== Loss Plot =====
axes[0].plot(epochs, train_losses, marker='o', label='Train Loss')
axes[0].plot(epochs, val_losses, marker='o', label='Val Loss')
axes[0].set_title("Train vs Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# ===== Accuracy Plot =====
axes[1].plot(epochs, train_accs, marker='o', label='Train Accuracy')
axes[1].plot(epochs, val_accs, marker='o', label='Val Accuracy')
axes[1].set_title("Train vs Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# prediction

In [ ]:
# Get a batch from the test DataLoader
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

# Model prediction (logits)
outputs = model(images)
print("Logits shape:", outputs.shape)   # (B, 2)

# Convert logits to predicted class labels
_, preds = torch.max(outputs, dim=1)
print("Predicted labels shape:", preds.shape)  # (B,)

# (Optional) Compare predictions with ground truth
print("Ground truth labels:", labels[:10].cpu().numpy())
print("Predicted labels:", preds[:10].cpu().numpy())
import torch.nn.functional as F

probs = F.softmax(outputs, dim=1)
print("Class probabilities:", probs[:5])

In [ ]:
inverse_transform = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

In [ ]:
# Get a batch from test loader
X, Y = next(iter(test_loader))
X = X.to(device)
Y = Y.to(device)

# Model prediction
with torch.no_grad():
    outputs = model(X)
    _, Y_pred = torch.max(outputs, dim=1)

# Inverse normalize images (back to original colors)
X_inv = torch.stack([inverse_transform(img) for img in X])

batch_size = X.shape[0]

# Create figure: 2 rows, batch_size columns
fig, axes = plt.subplots(2, batch_size, figsize=(3 * batch_size, 6))

label_map = {0: "No Cancer", 1: "Cancer"}

for i in range(batch_size):
    img = X_inv[i].permute(1, 2, 0).cpu().numpy()

    # Row 0: Ground Truth
    axes[0, i].imshow(img)
    axes[0, i].set_title(
        f"GT: {label_map[Y[i].item()]}",
        fontsize=14,
        fontweight="bold"
    )
    axes[0, i].axis("off")

    # Row 1: Prediction
    axes[1, i].imshow(img)
    axes[1, i].set_title(
        f"Pred: {label_map[Y_pred[i].item()]}",
        fontsize=14,
        fontweight="bold"
    )
    axes[1, i].axis("off")

# Row labels
axes[0, 0].set_ylabel("Ground Truth", fontsize=16, fontweight="bold")
axes[1, 0].set_ylabel("Prediction", fontsize=16, fontweight="bold")

plt.tight_layout()
plt.show()

# Model Prediction Validation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X, Y in test_loader:
        X = X.to(device)
        Y = Y.to(device)

        outputs = model(X)              # (B, 2)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(Y.cpu().numpy())

#  Metrics
acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)

print("Test Accuracy :", acc)
print("Test Precision:", precision)
print("Test Recall   :", recall)
print("Test F1-score :", f1)

print("\nConfusion Matrix:")
print(cm)